# Model-to-Data - Basic Usage and Overview

In this Notebook we will demonstrate how to use the eGSIM API to compare observed ground motion data to a set of ground motion models.

The objectives are as follows:

1) How to run a "rankings" analysis to retrieve fit-to-data scores for a selection of GMMs
   
2) Retreive the normalized random effects residuals (between-event $\delta B_E$ and within-event $\delta W$) for the same data set and ground motion models and plot these with respect to different predictor variables

3) Use the eGSIM API and pandas to make estimates of the site-to-site residuals $\delta S2S_S$


The data set used is adapted from a sub-set the Engineering Strong Motion (ESM) Flatfile (Lanzano et al. 2018) available from https://esm-db.eu/#/products/flat_file

```
Lanzano, G., Sgobba, S., Luzi, L., Puglia, R., Pacor, F., Felicetta, C., D’Amico, M., Cotton, F., & Bindi, D. (2019). The pan-European Engineering Strong Motion (ESM) flatfile: Compilation criteria and data statistics. Bulletin of Earthquake Engineering, 17(2), 561–582. https://doi.org/10.1007/s10518-018-0480-z
```

##### Tools for use here ...

In [ ]:
%matplotlib inline
import requests  # To make the eGSIM API request
from typing import List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # For some "prettier" plotting functionalities

In [ ]:
def get_residuals_from_egsim(
    flatfile_path: str,
    gmms: List,
    imts: List,
    data_format: str = "hdf",
    query_string: str = "",
    normalize: bool = True,
    likelihood: bool = False,
    ranking: bool = False
) -> Dict:
    """Retreive the residuals for the flatfile and the selected
    set of ground motion models and intensity measure types

    Args:
        flatfile_path: Local path to the selected flatfile
        gmms: List of ground motion models (OpenQuake class names)
        imts: List of intensity measure types (e.g. PGA, PGV, SA(0.1) etc.)
        plot_type: Column to return for x-values (e.g. mag, rrup, etc)
        query_string: Selection query to apply to the data

    Returns:
        json_response: Response of the POST request in json form (if successful)
                       or the response class (if unsuccessful)
    """
    assert data_format in ("hdf", "csv"), "Data format must be either 'hdf' or 'csv'"
    # Set the eGSIM URL to get the ground motion residuals
    egsim_url_residuals = "https://egsim.gfz-potsdam.de/api/query/residuals"
    # Retreive residuals for a given plot type
    parameters = {
            "model": gmms,
            "imt": imts,
            "format": data_format      
    }
    if query_string:
        parameters["data-query"] = query_string
    if not normalize:
        parameters["normalize"] = False
    if likelihood:
        parameters["likelihood"] = True
    if ranking:
        parameters["ranking"] = True

    with open(flatfile_path, "rb") as flatfile:
        files = {"flatfile": flatfile}
        try:
            # POST request for eGSIM
            response = requests.post(
                egsim_url_residuals,
                files=files,
                data=parameters
            )
            print(response.status_code)
            response.raise_for_status()
        except requests.exceptions.HTTPError as exc:
            code = exc.response.status_code
            print(response.text)
            msg = response.json()['message']
            print(f"HTTPError (code={code}): {msg}")
        except:
            print("Response failed - see status message")
            raise  
    
    if parameters['format'] == 'hdf':
        # `pd.read_hdf` works for HDF files on disk. Workaround:
        with pd.HDFStore(
                "data.h5",  # apparently unused for in-memory data
                mode="r",
                driver="H5FD_CORE",  # create in-memory file
                driver_core_backing_store=0,  # for safety, just in case
                driver_core_image=response.content) as store:
            dframe = store[list(store.keys())[0]]
    else:
        # use `pd.read_csv` with a BytesIO (file-like object) as input: 
        dframe = pd.read_csv(io.BytesIO(response.content), header=[0, 1, 2], index_col=0)        

    return dframe

# Log-likelihood Analysis and GMM Ranking

In this first analysis we choose four GMMs that we believe may be suitable candidates for our data set. Each GMM is defined using its OpenQuake class name

1) Bindi et al. (2014) - Using the Joyner-Boore Distance Metric (`BindiEtAl2014Rjb`)
2) Cauzzi et al. (2015) - Calibrated on Japanese/global data (`CauzziEtAl2014`)
3) Chiou & Youngs (2014) - NGA West GMM calibrated on Western US and Global Data (`ChiouYoungs2014`)
4) Kotha et al. (2020) (ESHM20) - Version of the Kotha et al. (2020) GMM originally fit to ESM data, with adjustments made for ESHM20 by Weatherill et al. (2020) (`KothaEtAl2020ESHM20`)

We also select five intensity measures: PGA, PGV, SA(0.2 s), SA(1.0 s), and SA(2.0 s).

Finally, we apply a filter to limit the comparison only to records:

* From earthquakes with magnitude (`mag`) $\geq 4.5$
* With rupture distances $\leq 300$ km
* With hypocenter depths $\leq 35$ km
* For stations that only have a measured Vs30

Initially we make this analysis online and download the results to an hdf5 file - which we place on the current path in `./egsim-residuals-ranking.hdf"`

In [ ]:
residual_data_with_ranking = pd.read_hdf("../data/downloads/egsim-residuals-ranking.hdf")
residual_data_with_ranking

In the above file we see a table with multiple measures for each GMM. This includes the `mean` and `stddev` of the total, between and within-event residuals for each IMT, the log-likelihood score and also the Euclidean distance ranking (`edr`). We'll limit the data to just the log-likelihood scores

In [ ]:
loglikelihood_columns = []
for col in residual_data_with_ranking.columns:
    if "loglikelihood" in col:
        loglikelihood_columns.append(col)
residual_data_with_ranking[loglikelihood_columns]

So the above analysis shows that when all the IMs are considered together the Kotha et al. (2020) ESHM20 GMM has the lowest log-likelihood score, implying the best overall fit. But we can also see that there are some differences between IMs.

Let's use the eGSIM API to now run the same analysis but with a greater number of spectral periods, to be able to visualise the trend in LLH with period.

In [ ]:
periods = [0.02, 0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0, 2.5, 3.0]
imts = ['PGV', 'PGA'] + [f"SA({per:.3f})" for per in periods]

gmms = {
    # OpenQuake Class Name: (Pretty name, plotting colour)
    "BindiEtAl2014Rjb": ("Bindi et al. (2014)", "tab:blue"),
    "CauzziEtAl2014": ("Cauzzi et al. (2015)", "tab:red"),
    "ChiouYoungs2014": ("Chiou & Youngs (2014)", "tab:green"),
    "KothaEtAl2020ESHM20": ("ESHM20", "k"),
}

query_string = "(rrup <= 300.0) & (mag >= 4.5) & (evt_depth <= 35.0) & (vs30measured == True)"

In [ ]:
# This may take a minute or so
rankings = get_residuals_from_egsim(
    "../data/flatfiles/esm_sample_for_demo.csv",  # Path to the flatfile
    gmms=list(gmms),  # List of GMMS
    imts=imts,  # List of IMTs
    query_string=query_string,  # The executed query string
    ranking=True  # Choose to return the ranking (scoring) metrics rather than residuals
)
rankings

Just for interest ... here is the full list of columns (metrics) returned:

In [ ]:
rankings.columns.to_list()

### Plot log-likehood ranking with period

With the results from the query we can produce a plot to see how the log-likelihood score changes with period

In [ ]:
# First, select the columns and extract the spectral period (if SA) or assign a period for plotting (PGV, PGA)
llh_columns = []
periods = []
for col in rankings.columns:
    if "loglikelihood" not in col:
        continue 
    i_m = col.split(" ")[0]
    if i_m == 'All_IMT':
        llh_columns.append(col)
        continue
    if i_m == "PGV":
        periods.append(0.003)
    elif i_m == "PGA":
        periods.append(0.006)
    else:
        periods.append(float(i_m.replace("SA(", "").replace(")", "")))
    llh_columns.append(col)
# Select just the loglikelihood columns
llh = rankings[llh_columns]
periods = np.array(periods)
idx = np.argsort(periods)
periods = periods[idx]

fig, ax = plt.subplots(1, 1, figsize=(8,8))
for gmm in llh.index:
    gmm_name, gmm_color = gmms[gmm]
    # First plot the PGV
    ax.plot(periods[0], llh.loc[gmm].to_numpy()[idx][0], "o", color=gmm_color, lw=2)
    # Then plot the spectra
    ax.plot(periods[1:], llh.loc[gmm].to_numpy()[idx][1:], "o-", color=gmm_color, label=gmm_name, lw=2)

# Make the axis pretty!
ax.grid(which="both")
ax.set_xscale("log")
ax.legend(loc="upper right", fontsize=14)
ax.set_xlabel("IMT / Period (s)", fontsize=16)
ax.set_ylabel("Log-likelihood", fontsize=16)
ax.set_xlim(0.0025, 10.0)
ax.set_xticks([0.003, 0.006, 0.01, 0.1, 1.0, 10.0], ["PGV", "PGA", "0.01", "0.1", "1.0", "10.0"])
ax.tick_params(labelsize=12, labelrotation=30)

# Compare Observations Against GMMs

The log-likelihood scoring provides a general score of the fit of the model to data, but for selecting GMMs it is more insightful to explore the trends with respect to the source, path and site parameters. The eGSIM platform can do this by returning the between-event ($\delta B_e$) and within-event ($\delta W$) residuals for a given data set an GMM. In the regression process these terms are Gaussian distributed such that $\delta B_e =\mathcal{N}\left( {0, \tau}\right)$ and $\delta W =\mathcal{N}\left({0, \phi}\right)$.

$\ln\left( {Y} \right) = \mu\left( {M, R, \mathbf{\theta}} \right) + \delta B_e + \delta W$

where $Y$ is the intensity measure, $ \mu\left( {M, R, \mathbf{\theta}} \right)$ the median ground motion, $M$ the magnitude, $R$ distance and $\theta$ the collection of additional input parameters required by the GMM. 

While the residuals themselves can be used for many purposes (we will come back to this later) most GMMs separate the total aleatory variability of the model into the between event variability and within-event variability component, which we will refer to as $\widehat{\tau}$ and $\widehat{\phi}$ respectively to indicate that these refer to the model variabilities rather than the observed variabilities. To understand the differences between the distribution of the residuals of the current data set with respect to the model variabilities, it is common to normalise the residuals such that:

$\delta B_e^{\star} = \frac{\delta B_e}{\widehat{\tau}}$

and

$\delta W^{\star} = \frac{\delta W}{\widehat{\phi}}$

The normalised residuals can be useful as a general metric as one would expect that a *well fitting* model would have distributions of normalised residuals that are close to a standard normal distribution $\mathcal{N}\left( {0, 1} \right)$. 

This is the *default* option for eGSIM, but it is possible to return the non-normalised residuals too.

In the following steps we will extract the normalised random effects residuals for a set of observations against the selected previously selected ground motions.

We will adjust the query now to consider events with $M_W \geq 4.0$, depth less than 35 km and with measured and inferred Vs30 (i.e. remove the `vs30measured == True` filter.

In [ ]:
# For the BBSpeed data the observations are limited to 0.05 s and greater
imts = ["PGV", "PGA", "SA(0.2)", "SA(2.0)"]

Call the eGSIM API - this may take a minute

In [ ]:
residuals_observations = get_residuals_from_egsim(
    "../data/flatfiles/esm_sample_for_demo.csv",  # Path to the flatfile
    gmms=list(gmms),  #
    imts=imts,
    query_string="(rrup <= 300.0) & (mag >= 4.0) & (evt_depth <= 35.0)",
    normalize=True,
)
residuals_observations

In [ ]:
residuals_observations.columns.to_list()

In [ ]:
def dataframe_for_seaborn(
        residuals: pd.DataFrame,
        gmms: Dict,
        selected_imts: List,
        residual_type: str,
        input_parameter: str,
        y_column_name: str = ""
) -> pd.DataFrame:
    """Converts a dataframe output from an eGSIM residuals query into
    a format useful for seaborn.

    Args:
        residuals: The dataframe output from the eGSIM query
        gmms: A dictionary of GMMs with each entry indicate by the OpenQuake
              class name and the value a tuple of a "pretty" name and plotting color
        selected_imts: The chosen IMTs for visualisation
        residual_type: The choice of "inter_event_residual", "intra_event_residual" or
                       "total_residual"
        input_paramter: The x-variable for the regression plots
        y_column_name: Optional name to call the y-variable in the dataframe

    Returns:
        Dataframe for seaborn visualization.
    """
    full_dataframe = []
    xcolumn = f"input {input_parameter}"
    for i, imt in enumerate(selected_imts):
        for j, (gmm, (gmm_label, gmm_color)) in enumerate(gmms.items()):
            ycolumn = f"{imt} {residual_type} {gmm}"
            xyvals = residuals[[xcolumn, ycolumn]].drop_duplicates(
                [xcolumn, ycolumn], inplace=False, ignore_index=True)
            if y_column_name:
                xyvals.rename(columns={ycolumn: y_column_name}, inplace=True, copy=False)
            xyvals["IMT"] = [imt] * xyvals.shape[0]
            xyvals["GMM"] = [gmm_label] * xyvals.shape[0]
            full_dataframe.append(xyvals)
    return pd.concat(full_dataframe, axis=0, ignore_index=True)

#### Plot the trend in between-event residual ($\delta B_e$) with respect to magnitude

In [ ]:
full_dataframe = dataframe_for_seaborn(
    residuals=residuals_observations,
    gmms = gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type = "inter_event_residual",
    input_parameter = "rupture_parameter mag",
    y_column_name="dBe"
)
with sns.plotting_context("paper", font_scale=1.5):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x="input rupture_parameter mag",
                           y="dBe",
                           col="GMM",
                           row="IMT",
                           facet_kws={"sharex": True, "sharey":True}, )
        fgrid.set(xlim=(4.0, 8.0), ylim=(-3.0, 3.0))
        fgrid.set_axis_labels(r"$M_W$", r"$\delta B_{e}$")
        fgrid.tight_layout()

#### Plot the trends in within-event residual ($\delta W_{es}$) with respect to rupture distance.

In [ ]:
full_dataframe = dataframe_for_seaborn(
    residuals=residuals_observations,
    gmms = gmms,
    selected_imts=["PGV", "PGA", "SA(0.2)", "SA(2.0)"],
    residual_type = "intra_event_residual",
    input_parameter = "distance_measure rrup",
    y_column_name="dW"
)

with sns.plotting_context("paper", font_scale=1.5):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x="input distance_measure rrup",
                           y="dW",
                           col="GMM",
                           row="IMT",
                           markers="x",
                           line_kws={"color": "tab:red",},
                           facet_kws={"sharex": True, "sharey":True}, )
        fgrid.set(xlim=(0, 300), ylim=(-3.5, 3.5))
        fgrid.set_axis_labels(r"$R_{RUP}$ (km)", r"$\delta W$")
        fgrid.tight_layout()

### Can we also extract dS2S?

The eGSIM API returns the between- and within-event residuals for the given flatfile, GMMs and IMTs. However, there may be applications where many sites have multiple observations, and as such the within-event residual can be decomposed into a systematic site-to-site residual term specific to each site ($\delta S2S_S$) and a remaining site-corrected within-event residual ($\delta W_{es}$). eGSIM doesn't (yet) return this term directly, but it is possible to retrieve using the non-normalized within-event residuals. 

In the first step, we will retrieve the non-normalized residuals ...

In [ ]:
residuals_observations = get_residuals_from_egsim(
    "../data/flatfiles/esm_sample_for_demo.csv",  # Path to the flatfile
    gmms=list(gmms),  #
    imts=imts,
    query_string="(rrup <= 300.0) & (mag >= 4.0) & (evt_depth <= 35.0)",
    normalize=False,
)
residuals_observations

To retrieve the $\delta S2S_S$ for each station we need the unique identifier for the station to be added to the residuals dataframe (this is not returned by eGSIM by default). We can retrive this from the original flatfile and applying the same filters ...

In [ ]:
# Load in from flatfile
observations = pd.read_csv( "../data/flatfiles/esm_sample_for_demo.csv", sep=",")
observations.query(
    "(rrup <= 300.0) & (mag >= 4.0) & (evt_depth <= 35.0)",
    inplace=True
)
observations

Then we can add the additional site parameters from the flatfile into the residuals dataframe. Here we will do this as a new column `input site_parameter stn_id`.

In [ ]:
# Can also add back columns from the original flatfile
residuals_observations["input site_parameter stn_id"] = observations["stn_id"].loc[residuals_observations.index]
residuals_observations

At this point we are interested in just the site properties and the within-event residuals, so we can work on a smaller dataframe with just these columns

In [ ]:
# Select just the within-event residuals and the site predictors
selection_columns = [
    'input site_parameter stn_id',
    'input site_parameter vs30',
    'input site_parameter vs30measured',
    'input site_parameter z1pt0',
    ]

for col in residuals_observations.columns:
    if ("intra_event_residual" in col) and (("PGA" in col) or ("PGV" in col) or ("SA(" in col)):
        selection_columns.append(col)

dwes = residuals_observations[selection_columns]
dwes

Now with the help of Pandas we can estimate $\delta S2S_S$ using the "frequentist" interpretation (i.e. the arithmetic mean of $\delta W$ for the site). We can also estimate the uncertainty in $\delta S2S_S$ by taking the sample standard deviation of $\delta W$ for the site, but we will focus just on the mean here. 

In [ ]:
# Frequentist interpretation where dS2Ss is the mean of dWes for a given station
# Group the within event residuals by station ID an apply the "mean" operation to the group
ds2s = dwes.groupby("input site_parameter stn_id").agg("mean")
# Now re-name the "intra_event_residual" to "dS2Ss" in the columns
column_mapping = dict([(orig, orig.replace("intra_event_residual", "dS2Ss")) for orig in ds2s.columns])
ds2s.rename(columns=column_mapping, inplace=True)
ds2s

If you would like to limit the analysis only to those stations with more than $N_{OBS}$ observations then these can be identified easily:

In [ ]:
n_obs = dwes.value_counts("input site_parameter stn_id")
n_obs = n_obs[n_obs >= 5]
n_obs

In [ ]:
ds2s_n10 = ds2s.loc[n_obs.index]
ds2s_n10

For the final plot, we will use the whole $\delta S2S_S$ data set though ...

Now we can use Seaborn to produce a more hierarchical plot with respect to the predictor variable $V_{S30}$. Here the data and the regression trends are separated to show those stations with a measured $V_{S30}$ and those with an inferred $V_{S30}$. 

In [ ]:
full_dataframe = []
xcolumn = "input site_parameter vs30"  # Choice of predictor variable
hue_column = "input site_parameter vs30measured"  # Longitudinal variable
selected_imts = ["PGV", "PGA", "SA(0.2)", "SA(2.0)"]
for i, imt in enumerate(selected_imts):
    for j, (gmm, (gmm_label, gmm_color)) in enumerate(gmms.items()):
        ycolumn = f"{imt} dS2Ss {gmm}"
        # Select the x-, y- and hue columns
        xyvals = ds2s[[xcolumn, ycolumn, hue_column]].drop_duplicates(
            [xcolumn, ycolumn],
            inplace=False,
            ignore_index=True
        )
        xyvals.rename(
            columns={ycolumn: "dS2Ss", hue_column: "Vs30 Measured"},
            inplace=True,
            copy=False
        )
        xyvals["IMT"] = [imt] * xyvals.shape[0]
        xyvals["GMM"] = [gmm_label] * xyvals.shape[0]
        full_dataframe.append(xyvals)
full_dataframe = pd.concat(full_dataframe, axis=0, ignore_index=True)
# Plot dS2S against Vs30, separating the measured and inferred sites
with sns.plotting_context("paper", font_scale=1.5):
    with sns.axes_style("darkgrid", ):
        fgrid = sns.lmplot(full_dataframe,
                           x=xcolumn,
                           y="dS2Ss",
                           hue="Vs30 Measured",
                           col="GMM",
                           row="IMT",
                           facet_kws={"sharex": True, "sharey":True}, )
        fgrid.set(xlim=(100, 1200), ylim=(-3.5, 3.5))
        fgrid.set_axis_labels(r"$V_{S30}$ (km)", r"$\delta S2S_S$")
        fgrid.tight_layout(pad=0.15)